# Projekt zaliczeniowy: Metody reprezentacji wiedzy

Temat: (wpiszcie temat z listy albo temat własny zatwierdzony przez prowadzącego)

Zespół: (imiona i nazwiska, 3 lub 4 osoby)

Notebook ma wykonywać się w Google Colab od początku do końca bez błędów (Środowisko wykonawcze, Uruchom wszystko). Oddajecie go z zapisanymi wynikami. Komórki z komentarzami są punktem wyjścia, możecie je dzielić, łączyć i dodawać nowe. Każdy wynik liczbowy i każdą tabelę skomentujcie w komórce tekstowej pod nią.

In [ ]:
# Instalacja bibliotek spoza domyślnego środowiska Colab. Dopiszcie tu wszystko, czego używacie.
%pip install -q rdflib owlrl owlready2 pgmpy

In [ ]:
import io
import re
import time
import shutil
import subprocess
import warnings

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

import rdflib
from rdflib import Graph, Namespace, Literal, URIRef, BNode, RDF, RDFS, OWL, XSD
from rdflib.namespace import SKOS
import owlrl
import owlready2

RANDOM_STATE = 42
T_START = time.time()

## 1. Zespół, dziedzina, cel systemu

Opiszcie dziedzinę i to, komu i do czego system ma służyć. Zawęźcie zakres jawnie: czego świadomie nie modelujecie. Uzasadnijcie, które części wiedzy wymagają którego formalizmu (wrócicie do tego w sekcji 8).

## 2. Pytania kompetencyjne

Co najmniej 12 pytań, w tym przynajmniej 4 wymagające wnioskowania (odpowiedź nie jest zapisana w danych wprost). Warto dodać 1 lub 2 pytania, których system celowo nie obsługuje, i wyjaśnić dlaczego.

In [ ]:
# Tabela pytań: id, treść, typ (bezpośrednie / wnioskowanie / nieobsługiwane), mechanizm odpowiedzi
CQ = pd.DataFrame([
    ("CQ1", "...", "bezpośrednie", "SPARQL"),
], columns=["id", "pytanie", "typ", "mechanizm"])
CQ

## 3. Źródła danych i licencje

Dla każdego źródła podajcie adres, licencję, datę pobrania i to, które informacje z niego bierzecie. Dane opracowane samodzielnie oznaczcie jako takie. Jeśli dane mogą wyrządzić szkodę, gdy są błędne (zdrowie, finanse, alergeny), napiszcie, jak je weryfikowaliście.

## 4. Ontologia: klasy, właściwości, aksjomaty, uzasadnienie decyzji modelowych

Wymagane minimum: 15 klas, 10 właściwości, etykiety (`rdfs:label`) i komentarze (`rdfs:comment`) dla klas. Klasy definiowane warunkami koniecznymi i wystarczającymi (`owl:equivalentClass`) dają reasonerowi coś do roboty. Opiszcie najważniejsze decyzje: co jest klasą, a co instancją, gdzie używacie `subClassOf`, a gdzie relacji części, po co są dziedziny i zakresy. Każdą decyzję musicie umieć obronić.

In [ ]:
# Ontologia w Turtle (wbudowana w kod albo wczytywana z publicznego adresu)
TBOX = """
@prefix ex: <http://example.org/projekt#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

"""
tbox = Graph().parse(data=TBOX, format="turtle")
print("trójek w TBox:", len(tbox))

## 5. Dane i budowa grafu wiedzy

Co najmniej 100 instancji. Pokażcie, z czego graf powstaje (tabela, plik, API) i jak wygląda po zbudowaniu: liczba klas, właściwości, instancji, trójek, rozkład instancji po klasach. Tu albo w osobnej podsekcji umieśćcie łączenie z zewnętrznym źródłem (np. Wikidata) wraz z liczbą udanych i błędnych powiązań. Zapytania do usług zewnętrznych wysyłajcie z nagłówkiem User-Agent i przygotujcie kopię wyników na wypadek niedostępności usługi.

In [ ]:
# Wczytanie danych i budowa grafu g (TBox + ABox)

In [ ]:
# Statystyki grafu: klasy, właściwości, instancje, trójki, klasy bez etykiety lub komentarza

In [ ]:
# Łączenie z zewnętrznym źródłem wiedzy i ocena powiązań

## 6. Zapytania SPARQL

Zapytania odpowiadające na pytania kompetencyjne. Pokażcie przynajmniej jedno zapytanie z agregacją i jedno z negacją (`FILTER NOT EXISTS`) i skomentujcie, jakie założenie o świecie przyjmuje to drugie.

In [ ]:
PREFIXES = """
PREFIX ex: <http://example.org/projekt#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
"""


def local(u):
    s = str(u)
    return re.split(r"[#/]", s)[-1] if isinstance(u, URIRef) else s


def sparql(graf, zapytanie):
    """Zwraca wynik zapytania SELECT jako DataFrame z nazwami lokalnymi zamiast pełnych adresów."""
    wynik = graf.query(PREFIXES + zapytanie)
    if wynik.type == "ASK":
        return wynik.askAnswer
    kolumny = [str(v) for v in wynik.vars]
    wiersze = [[local(x) if isinstance(x, URIRef) else (x.toPython() if x is not None else None) for x in row]
               for row in wynik]
    return pd.DataFrame(wiersze, columns=kolumny)


# Słownik zapytań dla pytań kompetencyjnych, np. Q["CQ1"] = """SELECT ..."""
Q = {}

## 7. Wnioskowanie

Użyjcie reasonera świadomie: pokażcie fakty i przynależności do klas, których nie było przed wnioskowaniem, a są po nim, i objaśnijcie co najmniej jeden taki wniosek (z jakich trójek i aksjomatów wynika). HermiT w owlready2 wymaga Javy. Komórka poniżej doinstalowuje ją w Colabie. Jeśli to się nie uda, korzystajcie z owlrl (profil OWL 2 RL), który działa w czystym Pythonie.

In [ ]:
if shutil.which("java") is None:
    !apt-get -qq install -y openjdk-17-jre-headless > /dev/null
JAVA_OK = shutil.which("java") is not None
if JAVA_OK:
    print(subprocess.run(["java", "-version"], capture_output=True, text=True).stderr.splitlines()[0])
else:
    print("Brak Javy: HermiT niedostępny, użyjcie owlrl.")

In [ ]:
# owlrl: domknięcie grafu w profilu OWL 2 RL i porównanie liczby trójek przed i po
# gi = Graph() + g
# owlrl.DeductiveClosure(owlrl.OWLRL_Semantics, axiomatic_triples=False, datatype_axioms=False).expand(gi)

In [ ]:
# HermiT: kontrola spójności i klasyfikacja (graf przekazany do owlready2 jako N-Triples)
# swiat = owlready2.World()
# onto = swiat.get_ontology("http://example.org/projekt").load(
#     fileobj=io.BytesIO(g.serialize(format="nt", encoding="utf-8")), format="ntriples")
# with onto:
#     owlready2.sync_reasoner_hermit(swiat, infer_property_values=True, debug=0)

In [ ]:
# Fakty, które pojawiły się dzięki wnioskowaniu, i objaśnienie co najmniej jednego z nich

## 8. Drugi formalizm i jego połączenie z grafem

Wybierzcie jeden: reguły (np. z negacją, z kontrolą stratyfikacji), ramy z wartościami domyślnymi, sieć bayesowska zasilana faktami z grafu albo skrypty. Uzasadnijcie, dlaczego ta część wiedzy wymaga innego formalizmu niż OWL. Formalizm ma czytać fakty z grafu albo zapisywać do niego wyniki, a nie działać obok. Jeśli używacie sieci bayesowskiej, opiszcie pochodzenie każdego prawdopodobieństwa warunkowego (dane, literatura, oszacowanie zespołu) i sprawdźcie wrażliwość wyniku na ich zmianę.

In [ ]:
# Implementacja drugiego formalizmu

In [ ]:
# Objaśnienia: łańcuch reguł albo wpływ kolejnych obserwacji na prawdopodobieństwo

## 9. Aplikacja: odpowiadanie na pytania, rekomendacja lub decyzja

System odpowiada na pytania zadawane po polsku albo wspiera decyzję. Przygotujcie zestaw kilkunastu lub więcej pytań testowych z oczekiwanymi odpowiedziami wpisanymi przed uruchomieniem systemu, łącznie z pytaniami trudnymi (odmiana, niejednoznaczność, pytania spoza zakresu). Podajcie trafność i omówcie błędy.

In [ ]:
# Aplikacja wiedzy

In [ ]:
# Zestaw ewaluacyjny i wyniki

## 10. Ocena: testy pytań kompetencyjnych, pokrycie, porównanie z podejściem bez jawnej wiedzy

Każde pytanie kompetencyjne ma oczekiwaną odpowiedź wpisaną w kodzie (ręcznie, nie wyliczoną z tych samych danych), a jedna komórka sprawdza wszystkie naraz. Funkcja `sprawdz_pytania` przyjmuje listę słowników z polami `id`, `pytanie`, `zapytanie` (tekst SPARQL, funkcja bez argumentów albo `None` dla pytań nieobsługiwanych) i `oczekiwane` (wartość, liczba albo zbiór).

Tu umieśćcie też analizę jakości ontologii: spójność sprawdzoną reasonerem, wykryte i poprawione błędy modelowania, uwagi o założeniu świata otwartego.

In [ ]:
def sprawdz_pytania(testy, graf):
    """Uruchamia testy pytań kompetencyjnych i zwraca tabelę wyników."""
    wiersze = []
    for t in testy:
        q = t["zapytanie"]
        if q is None:
            wynik, zgodnosc, opis = "nieobsługiwane", "nie dotyczy", "brak"
        else:
            try:
                if callable(q):
                    wynik, opis = q(), "funkcja " + q.__name__
                else:
                    df = sparql(graf, q)
                    wartosci = df.iloc[:, 0].tolist()
                    if isinstance(t["oczekiwane"], set):
                        wynik = set(wartosci)
                    else:
                        wynik = wartosci[0] if len(wartosci) == 1 else wartosci
                    opis = " ".join(q.split())[:60]
                if isinstance(wynik, (np.integer, float)) and float(wynik).is_integer():
                    wynik = int(wynik)
                zgodnosc = "tak" if wynik == t["oczekiwane"] else "NIE"
            except Exception as e:
                wynik, zgodnosc, opis = f"błąd: {type(e).__name__}: {e}", "NIE", str(q)[:60]
        wiersze.append({"id": t["id"], "pytanie": t["pytanie"], "zapytanie": opis,
                        "oczekiwany wynik": t["oczekiwane"], "wynik": wynik, "zgodność": zgodnosc})
    return pd.DataFrame(wiersze)


TESTY = [
    # {"id": "CQ1", "pytanie": "...", "zapytanie": Q["CQ1"], "oczekiwane": "..."},
]
# sprawdz_pytania(TESTY, gi)

In [ ]:
# Tabela pokrycia: które pytania są obsługiwane, które nie i dlaczego

In [ ]:
# Porównanie z podejściem bez jawnej reprezentacji wiedzy (ten sam zestaw pytań)

In [ ]:
# Analiza jakości ontologii: spójność, celowo wprowadzone błędy i ich wykrycie, założenie świata otwartego

## 11. Prywatność, przejrzystość, uczciwość

Jakie dane osobowe lub wrażliwe system przetwarza albo pozwala wywnioskować? Kto powinien mieć dostęp do której części grafu? Czy decyzje systemu mogą różnicować grupy osób? Przynajmniej jeden problem sprawdźcie liczbami albo zapytaniem, nie tylko opisem.

In [ ]:
# Sprawdzenie liczbowe wybranego problemu prywatności lub uczciwości

## 12. Wnioski, ograniczenia, dalsze kroki

Wnioski poparte liczbami z notebooka. Ograniczenia przyjętej reprezentacji: czego nie da się wyrazić, gdzie założenia upraszczają rzeczywistość, które dane są niepewne.

### Notatka dla odbiorcy nietechnicznego

Jedna strona po polsku, bez żargonu: co system robi, czego nie umie, komu i do czego może służyć.

## 13. Podział pracy w zespole

| Osoba | Odpowiadała za | Sekcje |
|---|---|---|
| (imię i nazwisko) | ... | ... |
| (imię i nazwisko) | ... | ... |
| (imię i nazwisko) | ... | ... |
| (imię i nazwisko) | ... | ... |

In [ ]:
# Zapis ontologii z danymi do pliku Turtle i łączny czas wykonania
# g.serialize("ontologia.ttl", format="turtle")
print(f"czas wykonania notebooka: {time.time() - T_START:.0f} s")